In [2]:
import pandas as pd
import numpy as np
from xbbg import blp
from datetime import datetime
from pathlib import Path
import time
import json

# Generate month-end trading dates from Jan 2010 to Dec 2025
dates = pd.date_range('2010-01-01', '2025-12-31', freq='BME')  # Business Month End
print(f"Total month-ends to pull: {len(dates)}")
print(f"First: {dates[0].strftime('%Y-%m-%d')}")
print(f"Last:  {dates[-1].strftime('%Y-%m-%d')}")

Total month-ends to pull: 192
First: 2010-01-29
Last:  2025-12-31


In [8]:
async def pull_membership(date_str):
    """Pull FTSE 250 constituents for a single date. Returns DataFrame or None."""
    for attempt in range(3):
        try:
            df = await blp.abds(
                'MCX Index',
                'INDX_MWEIGHT_HIST',
                END_DATE_OVERRIDE=date_str
            )
            # Convert to pandas if it's not already
            if hasattr(df, 'to_pandas'):
                df = df.to_pandas()
            elif not isinstance(df, pd.DataFrame):
                df = pd.DataFrame(df)
           
            if len(df) > 200:
                return df
            else:
                print(f"  Attempt {attempt+1}: only {len(df)} rows, retrying...")
                time.sleep(5)
        except Exception as e:
            print(f"  Attempt {attempt+1} error: {e}")
            time.sleep(10)
    return None

In [9]:
OUT_PATH = Path('../data/raw/membership_snapshots.parquet')
PROGRESS_PATH = Path('../data/raw/membership_progress.json')

# Resume support: load previously completed dates
completed = set()
all_rows = []

if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
        completed = set(progress.get('completed', []))
        print(f"Resuming: {len(completed)} dates already done")

if OUT_PATH.exists():
    existing = pd.read_parquet(OUT_PATH)
    all_rows.append(existing)
    print(f"Loaded {len(existing)} existing rows")

# Pull each month-end
for i, date in enumerate(dates):
    date_str = date.strftime('%Y%m%d')
   
    if date_str in completed:
        continue
   
    print(f"[{i+1}/192] Pulling {date.strftime('%Y-%m-%d')}...", end=' ')
   
    df = await pull_membership(date_str)
   
    if df is not None:
        df['snapshot_date'] = date
        all_rows.append(df)
        completed.add(date_str)
        print(f"OK — {len(df)} names")
    else:
        print(f"FAILED after 3 attempts")
   
    # Save progress every 10 snapshots
    if (i + 1) % 10 == 0:
        combined = pd.concat(all_rows, ignore_index=True)
        combined.to_parquet(OUT_PATH)
        with open(PROGRESS_PATH, 'w') as f:
            json.dump({'completed': list(completed)}, f)
        print(f"  [Checkpoint saved: {len(completed)} dates, {len(combined)} rows]")
   
    time.sleep(1)

# Final save
combined = pd.concat(all_rows, ignore_index=True)
combined.to_parquet(OUT_PATH)
with open(PROGRESS_PATH, 'w') as f:
    json.dump({'completed': list(completed)}, f)

print(f"\nDone! {len(completed)}/192 dates pulled, {len(combined)} total rows")
print(f"Saved to {OUT_PATH}")

[1/192] Pulling 2010-01-29... OK — 253 names
[2/192] Pulling 2010-02-26... OK — 253 names
[3/192] Pulling 2010-03-31... OK — 253 names
[4/192] Pulling 2010-04-30... OK — 253 names
[5/192] Pulling 2010-05-31... OK — 253 names
[6/192] Pulling 2010-06-30... OK — 254 names
[7/192] Pulling 2010-07-30... OK — 254 names
[8/192] Pulling 2010-08-31... OK — 254 names
[9/192] Pulling 2010-09-30... OK — 254 names
[10/192] Pulling 2010-10-29... OK — 254 names
  [Checkpoint saved: 10 dates, 2535 rows]
[11/192] Pulling 2010-11-30... OK — 254 names
[12/192] Pulling 2010-12-31... OK — 254 names
[13/192] Pulling 2011-01-31... OK — 254 names
[14/192] Pulling 2011-02-28... OK — 254 names
[15/192] Pulling 2011-03-31... OK — 254 names
[16/192] Pulling 2011-04-29... OK — 254 names
[17/192] Pulling 2011-05-31... OK — 254 names
[18/192] Pulling 2011-06-30... OK — 254 names
[19/192] Pulling 2011-07-29... OK — 254 names
[20/192] Pulling 2011-08-31... OK — 254 names
  [Checkpoint saved: 20 dates, 5075 rows]
[21/1

In [10]:
# Load the membership panel
panel = pd.read_parquet('../data/raw/membership_snapshots.parquet')
print(f"Total rows: {len(panel):,}")
print(f"Columns: {list(panel.columns)}")
print(f"\nFirst few rows:")
print(panel.head(10))

Total rows: 48,210
Columns: ['ticker', 'field', 'Index Member', 'Percent Weight', 'snapshot_date']

First few rows:
      ticker              field Index Member  Percent Weight snapshot_date
0  MCX Index  INDX_MWEIGHT_HIST  1218069D LN        0.092829    2010-01-29
1  MCX Index  INDX_MWEIGHT_HIST  1334987D LN        0.156177    2010-01-29
2  MCX Index  INDX_MWEIGHT_HIST  1561649D LN        0.301515    2010-01-29
3  MCX Index  INDX_MWEIGHT_HIST  1655637D LN        1.020425    2010-01-29
4  MCX Index  INDX_MWEIGHT_HIST  1707299D LN        0.277235    2010-01-29
5  MCX Index  INDX_MWEIGHT_HIST  1816878D LN        0.533091    2010-01-29
6  MCX Index  INDX_MWEIGHT_HIST  1926157D LN        0.645628    2010-01-29
7  MCX Index  INDX_MWEIGHT_HIST  2627525D LN        0.238709    2010-01-29
8  MCX Index  INDX_MWEIGHT_HIST  3572335Q LN        0.368127    2010-01-29
9  MCX Index  INDX_MWEIGHT_HIST       3IN LN        0.332028    2010-01-29


In [11]:
# The actual tickers are in 'Index Member'
# Normalise to Bloomberg's standard format: "XXX LN Equity"
def normalise(t):
    t = str(t).strip().upper()
    if t.endswith(' EQUITY'):
        return t
    if ' LN' in t:
        return t.split(' LN')[0] + ' LN Equity'
    return t + ' LN Equity'

# Clean up the panel
membership = panel[['snapshot_date', 'Index Member', 'Percent Weight']].copy()
membership.columns = ['snapshot_date', 'ticker_raw', 'weight']
membership['ticker'] = membership['ticker_raw'].apply(normalise)

# Master list: every unique ticker ever in the index
master = membership['ticker'].drop_duplicates().sort_values().reset_index(drop=True)
print(f"Total unique tickers ever in FTSE 250 (2010-2025): {len(master)}")
print(f"\nFirst 10:")
print(master.head(10).to_string())
print(f"\nLast 10:")
print(master.tail(10).to_string())

# Count how many have the 'D' suffix (delisted)
delisted = master[master.str.contains(r'\dD LN', regex=True)]
print(f"\nDelisted tickers (numeric+D format): {len(delisted)}")
print(f"Still-active tickers: {len(master) - len(delisted)}")

Total unique tickers ever in FTSE 250 (2010-2025): 641

First 10:
0    0966088D LN Equity
1    1218069D LN Equity
2    1245147D LN Equity
3    1334987D LN Equity
4    1479704D LN Equity
5    1561649D LN Equity
6    1638414D LN Equity
7    1655637D LN Equity
8    1707299D LN Equity
9    1777149D LN Equity

Last 10:
631     WPP LN Equity
632     WSM LN Equity
633    WTAN LN Equity
634     WWH LN Equity
635     XAR LN Equity
636     XCH LN Equity
637     XPP LN Equity
638     XPS LN Equity
639     ZIG LN Equity
640     ZPG LN Equity

Delisted tickers (numeric+D format): 21
Still-active tickers: 620


In [12]:
# Save master ticker list
master.to_frame('ticker').to_parquet('../data/raw/master_tickers.parquet')
print(f"Saved master ticker list: {len(master)} tickers")

# Save cleaned membership panel
membership[['snapshot_date', 'ticker', 'weight']].to_parquet(
    '../data/raw/membership_panel.parquet'
)
print(f"Saved membership panel: {len(membership)} rows")

# Quick sanity check: names per snapshot
names_per_month = membership.groupby('snapshot_date')['ticker'].nunique()
print(f"\nNames per monthly snapshot:")
print(f"  Min:    {names_per_month.min()}")
print(f"  Max:    {names_per_month.max()}")
print(f"  Median: {names_per_month.median()}")
print(f"  Any below 240? {(names_per_month < 240).sum()} months")

Saved master ticker list: 641 tickers
Saved membership panel: 48210 rows

Names per monthly snapshot:
  Min:    250
  Max:    255
  Median: 250.0
  Any below 240? 0 months
